# 00 · The PMD infrastructure: Ontodocker and the Dataportal with praeco

**Goal.** Show the services of the Platform MaterialDigital that the tensile series is built on, used directly through the Python client [praeco](https://github.com/materialdigital/praeco): find research graphs on Ontodocker, ask them questions with SPARQL, combine what belongs together into one publication artifact, harvest the publication metadata from the graph itself, review it, and publish the artifact as a record on the MaterialDigital Dataportal. [Notebook 06](06-tensile-results-publication.ipynb) wraps the same calls in workflow nodes.

**Audience.** All readers; the optional start for anyone new to Ontodocker and the Dataportal.

**Prerequisites.**
- Python and a first idea of RDF (subject, predicate, object) and SPARQL.
- The `pmd-demonstrator` environment from `environment.yml`, which installs praeco and the package `pmd_tensile`.
- `ONTODOCKER_ADDRESS` (and `ONTODOCKER_TOKEN`, if the instance needs one) set in the environment before Jupyter starts; for publishing also `DATAPORTAL_TOKEN`. None of them is printed here.

**Reading paths.** MSE domain experts: [05](05-tensile-analysis-overview.ipynb) → [02](02-tensile-method.ipynb) → [01](01-tensile-data-from-graphs.ipynb) → [06](06-tensile-results-publication.ipynb). Ontologists and RDF experts: [01](01-tensile-data-from-graphs.ipynb) → [04](04-tensile-workflow-knowledge-graph.ipynb) → [05](05-tensile-analysis-overview.ipynb) → [06](06-tensile-results-publication.ipynb). Workflow developers: [03](03-tensile-workflow-building.ipynb) → [04](04-tensile-workflow-knowledge-graph.ipynb) → [02](02-tensile-method.ipynb) → [05](05-tensile-analysis-overview.ipynb) → [06](06-tensile-results-publication.ipynb). This notebook (**00**) is the optional start of each.

| Term | Meaning |
| --- | --- |
| Ontodocker | a triple store of the platform; each *dataset* is one RDF graph with a SPARQL endpoint |
| Dataportal | the platform's catalogue (CKAN); a *record* has metadata and *assets* (uploaded files or links) |
| research graph | the RDF description of the ten tensile tests on S355 steel, in two ontology generations (PMDco 2 and PMDco 3, see [notebook 01](01-tensile-data-from-graphs.ipynb)) |
| harvester | praeco's `harvest_publication_metadata_from_rdf`: reads title, description, keywords and more from a graph, with the triples as evidence |

*Execution class: network-read* with the default `publish_records = []`: queries and downloads from Ontodocker, public reads of the Dataportal; files are written to a temporary directory only. With record names in `publish_records` it is *remote-write*: it creates those Dataportal records (section 5), which must not exist (a deleted record's name can be reused).

## Setup

The next cell is tagged `parameters` for papermill. `publish_records` names the records to create, in that order; empty, the notebook only shows what would be published.

In [1]:
publish_records = []  # e.g. ["pmdco3-tto-recombined"]; empty: read only
publication_date = None  # ISO date for both records; None: their reviewed dates below
owner_org = "pmd"

In [2]:
import datetime
import hashlib
import json
import sys
import tempfile
from pathlib import Path

import pandas as pd
import rdflib
from IPython.display import display
from praeco import Person, harvest_publication_metadata_from_rdf
from praeco.exceptions import HttpError
from praeco.rdf_metadata import IncompleteHarvestError
from praeco.services.dataportal.metadata import DataportalMetadata

import pmd_tensile
from pmd_tensile import sources

pd.set_option("display.max_colwidth", 120)
REPOSITORY = Path(pmd_tensile.__file__).resolve().parents[1]  # the editable install of this repository
sys.path.insert(0, str(REPOSITORY))
from query_collection import S355queries  # noqa: E402  (the repository's query set)

workdir = tempfile.TemporaryDirectory(prefix="pmd-tour-")  # removed in the last cell
ontodocker = sources.ontodocker_client()  # praeco's OntodockerClient; address and token from the environment
dataportal = sources.dataportal_client()  # praeco's DataportalClient, anonymous: public reads only
retrieved = pd.Timestamp.now(tz="Europe/Berlin").floor("s")
print("data retrieved:", retrieved)

data retrieved: 2026-11-10 22:37:54+01:00


## 1. What is available?

`datasets.list()` returns the names of the hosted graphs. The PMD research graphs of the tensile tests are split by orientation group, once per ontology generation:

In [3]:
names = ontodocker.datasets.list()
tensile = sorted(name for name in names if name.startswith(("pmdco2_tto_example", "pmdco3_tto_example")))
print(f"{len(names)} datasets hosted; tensile-test graphs:")
display(pd.DataFrame({"dataset": tensile, "generation": ["PMDco 2" if n.startswith("pmdco2") else "PMDco 3" for n in tensile]}))

7 datasets hosted; tensile-test graphs:


,dataset,generation
0,pmdco2_tto_example_diagonal,PMDco 2
1,pmdco2_tto_example_parallel,PMDco 2
2,pmdco2_tto_example_perpendicular,PMDco 2
3,pmdco3_tto_example_diagonal,PMDco 3
4,pmdco3_tto_example_parallel,PMDco 3
5,pmdco3_tto_example_perpendicular,PMDco 3
6,pmdco3_tto_example_results,PMDco 3


## 2. What does it mean?

A SPARQL query asks a graph a question in terms of its ontology. The queries of `query_collection.py` are written for PMDco 2; here they ask the diagonal group which material each test used, in which orientation the specimen was cut, and where its raw curve is.

In [4]:
def ask(dataset, query):
    return ontodocker.sparql.query_df(dataset, query.query, query.query_vars)


def test_label(iri):
    return str(iri).rsplit("/", 1)[-1].replace("pmdao-tto-tt-S355-", "test ").removesuffix("_process")


diagonal = "pmdco2_tto_example_diagonal"
material, orientation, curves = (ask(diagonal, q) for q in (S355queries.material_designation(), S355queries.orientation(),
                                                           S355queries.csv_url()))
answers = material.merge(orientation, on="uri").merge(curves, on="uri")
display(answers.assign(uri=answers["uri"].map(test_label)).rename(columns={"uri": "test"}))
print(S355queries.orientation().query.strip())

,test,materialDesignation,cut_orientation,url
0,test 10,S355,diagonal to rolling direction,https://data.pmds-mpcdf-2.pyiron.com/Zd3.csv
1,test 9,S355,diagonal to rolling direction,https://data.pmds-mpcdf-2.pyiron.com/Zd2.csv


PREFIX pmd: <https://w3id.org/pmd/co/>
        SELECT distinct (?p AS ?uri) (?rollingDirection AS ?cut_orientation)
        WHERE {
            ?s a pmd:TestPiece .
            ?p a pmd:TensileTest .
            ?p pmd:input ?s .
            ?p pmd:characteristic ?characteristic .
            ?characteristic a pmd:MaterialRelated .
            ?characteristic pmd:value ?rollingDirection .
        }
        ORDER BY ?p


The same question asked of all three PMDco 2 groups:

In [5]:
pmdco2 = [f"pmdco2_tto_example_{group}" for group in ("parallel", "perpendicular", "diagonal")]
per_group = pd.concat([ask(name, S355queries.orientation()).assign(dataset=name) for name in pmdco2], ignore_index=True)
display(per_group.groupby(["dataset", "cut_orientation"]).size().rename("tests").to_frame())

,,tests
dataset,cut_orientation,
pmdco2_tto_example_diagonal,diagonal to rolling direction,2
pmdco2_tto_example_parallel,in rolling direction,4
pmdco2_tto_example_perpendicular,perpendicular to rolling direction,4


## 3. What belongs together?

The three PMDco 2 groups describe one test series, so they are published as one artifact: each graph is downloaded as Turtle and the three are merged. The merge is compared with the committed artifact `datasets/pmdco2_tto_recombined.ttl`: the same number of triples, and the same triples apart from those with blank nodes (the column lists of the CSVW tables), whose names differ from file to file. (A full isomorphism check is very slow on graphs with about a thousand blank-node triples.) The PMDco 3 artifact is the committed full file `datasets/pmdco3_tto_example.ttl`; its three hosted slices are cut from it (notebook 01).

In [6]:
folder = Path(workdir.name)
combined = rdflib.Graph()
for name in pmdco2:
    path = ontodocker.datasets.download_turtle(name, folder / f"{name}.ttl")
    combined.parse(path, format="turtle")
artifact_pmdco2 = folder / "pmdco2_tto_recombined.ttl"
combined.serialize(destination=artifact_pmdco2, format="turtle")
committed = rdflib.Graph().parse(REPOSITORY / "datasets" / "pmdco2_tto_recombined.ttl", format="turtle")
artifacts = {"pmdco2-tto-recombined": artifact_pmdco2, "pmdco3-tto-recombined": REPOSITORY / "datasets" / "pmdco3_tto_example.ttl"}
def without_blank_nodes(graph):
    return {t for t in graph if not any(isinstance(term, rdflib.BNode) for term in t)}


print(f"merged {len(pmdco2)} graphs: {len(combined)} triples (committed artifact: {len(committed)}); "
      "triples without blank nodes equal:", without_blank_nodes(combined) == without_blank_nodes(committed))

merged 3 graphs: 3352 triples (committed artifact: 3352); triples without blank nodes equal: True


## 4. Which metadata can be reused?

The harvester reads the publication metadata from each artifact and keeps the triples it used as evidence. A field is a *candidate* when the graph states it directly, a *suggestion* when it is derived (e.g. a description built from the tests and the standard it finds), and a *review item* when a person must decide. It refuses to produce publication metadata while review items are open:

In [7]:
FIELDS = ("title", "description", "keywords", "publication_date", "license", "creators")
harvests = {record: harvest_publication_metadata_from_rdf(path).select_subject_automatically() for record, path in artifacts.items()}
rows = []
for record, harvest in harvests.items():
    for field in FIELDS:
        review = getattr(harvest, field)
        if field == "creators":  # creators are observed in the graph, not suggested
            found = f"{len(review.observations)} observation(s) in the graph"
        else:
            found = "; ".join(str(item.value) for item in review.candidates + review.suggestions)[:120] or "–"
        rows.append({"record": record, "field": field, "status": review.status, "found": found})
display(pd.DataFrame(rows).set_index(["record", "field"]))
for record, harvest in harvests.items():
    try:
        harvest.to_publication_metadata()
    except IncompleteHarvestError as error:
        print(f"{record}: not yet publishable: {error}")

status  \
record                field                          
pmdco2-tto-recombined title               resolved   
                      description          missing   
                      keywords             missing   
                      publication_date     missing   
                      license              missing   
                      creators          unresolved   
pmdco3-tto-recombined title               resolved   
                      description          missing   
                      keywords             missing   
                      publication_date     missing   
                      license              missing   
                      creators             missing   

                                                                                                                found  
record                field                                                                                            
pmdco2-tto-recombined title                                          Tensile Test Ontology (TTO) data mapping example  
                      description       Data from 10 tensile tests. Material: S355. Test standard: DIN EN ISO 6892-1.  
                      keywords                                                               ('S355', 'tensile test')  
                      publication_date                                                                              –  
                      license                                                                                       –  
                      creators                                                          4 observation(s) in the graph  
pmdco3-tto-recombined title                       TTO/PMDco 3.0 A-Box mapping example of S355 steel tensile test data  
                      description       Data from 10 tensile tests. Material: S355. Test standard: DIN EN ISO 6892-1.  
                      keywords                                                               ('S355', 'tensile test')  
                      publication_date                                                                     2026-04-16  
                      license                                                                                       –  
                      creators                                                          0 observation(s) in the graph

pmdco2-tto-recombined: not yet publishable: description requires review.; Creator <https://orcid.org/0000-0001-7192-7143> has unresolved kind or name.; Creator <https://orcid.org/0000-0002-3717-7104> has unresolved kind or name.; Creator <https://orcid.org/0000-0002-7094-5371> has unresolved kind or name.; Creator <https://orcid.org/0000-0003-4971-3645> has unresolved kind or name.
pmdco3-tto-recombined: not yet publishable: description requires review.; creators requires a non-empty reviewed list.


The reviewer's decisions, the same for both records unless stated:

- **accept** the suggested description, derived from the graph;
- **keep** the titles of the existing records;
- **supply** the creators: the four people named in the graphs, without ORCIDs, because the PMDco 2 graph does not pair names with ORCIDs;
- **supply** the license, CC-BY-4.0, which no graph states;
- **supply** the keywords: those of the records as first published;
- **supply** the publication date: that of the records as first published, unless `publication_date` is set.

The Dataportal record adds a name, the owning organisation, the visibility and audit fields (`extras`): the artifact's SHA-256, size and triple count, the CSV links, how the metadata was harvested, and, where the graph names creators by ORCID without pairing them with names, the list of those ORCIDs.

In [8]:
existing = {}
for record in artifacts:
    try:
        existing[record] = dataportal.datasets.show(record).raw
    except HttpError as error:  # a public read; a missing or deleted record has no previous values
        print(f"{record}: no readable record (status {error.status_code})")
CREATORS = ("Bernd Bayerlein", "Jörg Waitelonis", "Markus Schilling", "Philipp von Hartrott")
TITLES = {"pmdco2-tto-recombined": "Recombined S355 tensile test RDF dataset",
          "pmdco3-tto-recombined": "Recombined S355 tensile test RDF dataset (PMDco 3)"}
KEYWORDS = {"pmdco2-tto-recombined": ('CSVW', 'Ontodocker', 'PMD', 'Platform MaterialDigital', 'RDF', 'S355', 'Turtle', 'tensile test'),
            "pmdco3-tto-recombined": ('CSVW', 'Ontodocker', 'PMD', 'PMDco 3', 'Platform MaterialDigital', 'RDF', 'S355', 'TTO', 'Turtle', 'tensile test')}
ISSUED = {"pmdco2-tto-recombined": "2026-09-29", "pmdco3-tto-recombined": "2026-11-09"}
CSVW_URL = rdflib.URIRef("http://www.w3.org/ns/csvw#url")
DC_CREATOR = rdflib.URIRef("http://purl.org/dc/elements/1.1/creator")


def reviewed_record(record, harvest, path):
    issued = publication_date or ISSUED[record]
    keywords = KEYWORDS[record]
    reviewed = harvest.accept_suggestion(harvest.description.suggestions[0]).with_overrides(
        title=TITLES[record], creators=tuple(Person(name=name) for name in CREATORS), license="CC-BY-4.0",
        keywords=keywords, publication_date=datetime.date.fromisoformat(issued),
    )
    graph = rdflib.Graph().parse(path, format="turtle")
    data = path.read_bytes()
    csv_urls = sorted({str(url) for url in graph.objects(None, CSVW_URL)})
    extras = {"sha256": hashlib.sha256(data).hexdigest(), "file_size_bytes": str(len(data)),
              "rdf_triple_count": str(len(graph)), "csv_resource_urls": json.dumps(csv_urls, separators=(",", ":")),
              "source_file": f"datasets/{path.name}" if record.startswith("pmdco3") else "merged from " + ", ".join(pmdco2),
              "harvested_by": "praeco harvest_publication_metadata_from_rdf (reviewed)"}
    orcids = sorted({str(c) for c in graph.objects(None, DC_CREATOR) if str(c).startswith("https://orcid.org/")})
    if orcids:  # kept for audit: the graph names these ORCIDs but does not pair them with the creators' names
        extras["creator_orcids"] = json.dumps(orcids, separators=(",", ":"))
    return DataportalMetadata(metadata=reviewed.to_publication_metadata(), name=record, owner_org=owner_org,
                              private=False, extras=extras), csv_urls


reviewed = {record: reviewed_record(record, harvests[record], artifacts[record]) for record in artifacts}
display(pd.DataFrame({record: {"title": m.metadata.title, "description": m.metadata.description,
                               "creators": ", ".join(p.name for p in m.metadata.creators), "license": m.metadata.license,
                               "keywords": ", ".join(m.metadata.keywords), "publication date": str(m.metadata.publication_date),
                               "CSV links": len(csv_urls), "artifact SHA-256": m.extras["sha256"][:16] + "…"}
                      for record, (m, csv_urls) in reviewed.items()}))

,pmdco2-tto-recombined,pmdco3-tto-recombined
title,Recombined S355 tensile test RDF dataset,Recombined S355 tensile test RDF dataset (PMDco 3)
description,Data from 10 tensile tests. Material: S355. Test standard: DIN EN ISO 6892-1.,Data from 10 tensile tests. Material: S355. Test standard: DIN EN ISO 6892-1.
creators,"Bernd Bayerlein, Jörg Waitelonis, Markus Schilling, Philipp von Hartrott","Bernd Bayerlein, Jörg Waitelonis, Markus Schilling, Philipp von Hartrott"
license,CC-BY-4.0,CC-BY-4.0
keywords,"CSVW, Ontodocker, PMD, Platform MaterialDigital, RDF, S355, Turtle, tensile test","CSVW, Ontodocker, PMD, PMDco 3, Platform MaterialDigital, RDF, S355, TTO, Turtle, tensile test"
publication date,2026-09-29,2026-11-09
CSV links,10,11
artifact SHA-256,d0167d62ac5f5103…,ed8af84426ee23c1…


## 5. Publishing

Each record named in `publish_records` is created, in that order, from its reviewed metadata, the artifact is uploaded as an RDF asset, and every CSV file the graph links is added as a link asset. Creation stops if a record of that name exists and is not deleted. The token comes from the environment through `sources.dataportal_client(write=True)`; the cell prints no credential.

In [9]:
unknown = sorted(set(publish_records) - set(reviewed))
if unknown:
    raise ValueError(f"unknown records: {unknown}")
if publish_records:
    writer = sources.dataportal_client(write=True)
    for record in publish_records:
        metadata, csv_urls = reviewed[record]
        state = existing.get(record, {}).get("state")
        if state not in (None, "deleted"):
            raise RuntimeError(f"{record} exists ({state}); delete it first")
        created = writer.datasets.create(metadata)
        writer.assets.upload_rdf(created, artifacts[record], name=artifacts[record].name,
                                 description="Combined RDF/Turtle scientific description of the ten tensile tests.")
        for url in csv_urls:
            writer.assets.create_url(created, url=url, name=url.rsplit("/", 1)[-1], format="CSV",
                                     description="CSV resource referenced by the scientific graph.")
        print(f"created {record}: 1 artifact, {len(csv_urls)} CSV links")
for record, (metadata, _) in reviewed.items():
    if record not in publish_records:
        print(f"{record}: preview only; payload fields:", ", ".join(sorted(metadata.to_payload())))

pmdco2-tto-recombined: preview only; payload fields: creator, extras, issued, license_id, name, notes, owner_org, private, tags, title
pmdco3-tto-recombined: preview only; payload fields: creator, extras, issued, license_id, name, notes, owner_org, private, tags, title


## 6. What can others access?

A public, anonymous read of each record (for a record not created in this run, of the existing one, so the table shows what a new record would change): are its metadata the reviewed ones, are its CSV links those of the graph, and is the uploaded artifact byte for byte the local one? The Dataportal also adds, for each CSV link, an RDF and a JSON-LD description of its own; they are counted separately.

In [10]:
checks = {}
for record, (metadata, csv_urls) in reviewed.items():
    raw = dataportal.datasets.show(record).raw
    resources = raw["resources"]
    artifact = next((r for r in resources if r["name"] == artifacts[record].name), None)
    downloaded = dataportal.request("GET", artifact["url"]).content if artifact else b""
    checks[record] = {
        "title": raw["title"] == metadata.metadata.title,
        "description": raw["notes"] == metadata.metadata.description,
        "license": raw["license_id"] == metadata.metadata.license,
        "creators": [c["name"] for c in raw.get("creator") or []] == [p.name for p in metadata.metadata.creators],
        "keywords": sorted(t["name"] for t in raw["tags"]) == sorted(metadata.metadata.keywords),
        "CSV links = csvw:url of the graph": sorted(r["url"] for r in resources if r["format"] == "CSV") == csv_urls,
        "artifact = local file (SHA-256)": hashlib.sha256(downloaded).hexdigest() == metadata.extras["sha256"],
        "resources added by the Dataportal": sum(1 for r in resources if r["format"] not in ("CSV", "ttl")),
    }
display(pd.DataFrame(checks))

,pmdco2-tto-recombined,pmdco3-tto-recombined
title,True,True
description,False,True
license,True,True
creators,True,True
keywords,True,True
CSV links = csvw:url of the graph,True,True
artifact = local file (SHA-256),True,True
resources added by the Dataportal,20,21


## Summary

- Ontodocker hosts the research graphs per group; praeco lists them and runs SPARQL queries on each (sections 1 and 2).
- The three PMDco 2 groups merge into one artifact, identical as an RDF graph to the committed one (section 3).
- The harvester derives the description from each graph and stops until a person supplies creators, license and keywords (section 4).
- A record is created only from reviewed metadata, never over an existing one; a public read shows what others get (sections 5 and 6).

Notebook 06 does the same for the computed results, with workflow nodes that wrap these calls.

In [11]:
workdir.cleanup()